# 🏥 Patient Readmission Prediction
## Step 3: Model Training & Evaluation

Models: Logistic Regression → Random Forest → XGBoost
Metrics: Accuracy, ROC-AUC, Precision, Recall, F1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, roc_curve, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded!')

## 3.1 Load Processed Data

In [ ]:
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test  = data['X_test']
y_train = data['y_train']
y_test  = data['y_test']
feature_names = data['feature_names']

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3.2 Model 1: Logistic Regression (Baseline)

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

lr_preds = lr.predict(X_test_sc)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, lr_preds, target_names=['Not Readmitted', 'Readmitted']))
print(f'ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}')

## 3.3 Model 2: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, 
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, rf_preds, target_names=['Not Readmitted', 'Readmitted']))
print(f'ROC-AUC: {roc_auc_score(y_test, rf_proba):.4f}')

## 3.4 Model 3: Gradient Boosting (XGBoost alternative)

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, max_depth=4, 
                                 learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

gb_preds = gb.predict(X_test)
gb_proba = gb.predict_proba(X_test)[:, 1]

print('=== Gradient Boosting ===')
print(classification_report(y_test, gb_preds, target_names=['Not Readmitted', 'Readmitted']))
print(f'ROC-AUC: {roc_auc_score(y_test, gb_proba):.4f}')

## 3.5 Model Comparison

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = {
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'Accuracy': [
        accuracy_score(y_test, lr_preds),
        accuracy_score(y_test, rf_preds),
        accuracy_score(y_test, gb_preds)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, lr_proba),
        roc_auc_score(y_test, rf_proba),
        roc_auc_score(y_test, gb_proba)
    ],
    'Recall': [
        recall_score(y_test, lr_preds),
        recall_score(y_test, rf_preds),
        recall_score(y_test, gb_preds)
    ],
    'F1': [
        f1_score(y_test, lr_preds),
        f1_score(y_test, rf_preds),
        f1_score(y_test, gb_preds)
    ]
}

results_df = pd.DataFrame(results).round(4)
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
metrics = ['Accuracy', 'ROC-AUC', 'Recall', 'F1']
x = np.arange(len(metrics))
width = 0.25
colors = ['#5C6BC0', '#26A69A', '#EF5350']

for i, (model, color) in enumerate(zip(results['Model'], colors)):
    vals = [results_df[m].iloc[i] for m in metrics]
    axes[0].bar(x + i*width, vals, width, label=model, color=color, alpha=0.85)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Model Performance Comparison', fontsize=13)
axes[0].legend(fontsize=9)

# ROC Curves
for proba, label, color in [
    (lr_proba, f'LR (AUC={roc_auc_score(y_test,lr_proba):.3f})', '#5C6BC0'),
    (rf_proba, f'RF (AUC={roc_auc_score(y_test,rf_proba):.3f})', '#26A69A'),
    (gb_proba, f'GB (AUC={roc_auc_score(y_test,gb_proba):.3f})', '#EF5350')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[1].plot(fpr, tpr, label=label, color=color, lw=2)

axes[1].plot([0,1],[0,1],'k--', alpha=0.3)
axes[1].set_title('ROC Curves', fontsize=13)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_names)
top15 = importances.nlargest(15).sort_values()

plt.figure(figsize=(10, 7))
bars = plt.barh(top15.index, top15.values, color='#26A69A', alpha=0.85)
plt.xlabel('Feature Importance Score')
plt.title('Top 15 Features — Random Forest', fontsize=14)
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features predicting readmission:')
for feat, score in importances.nlargest(5).items():
    print(f'  {feat}: {score:.4f}')

## 3.7 Save Best Model

In [ ]:
# Save the best model (Random Forest) and scaler
with open('../data/best_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('✅ Best model saved: data/best_model.pkl')
print('✅ Scaler saved: data/scaler.pkl')
print('\nProceed to Streamlit app: app.py')